# **Attach a Microsoft Foundry Agent through its project** via Responses API

This notebook shows how to invoke a **non‑published Agent v2** inside a **Foundry project** using the **Project OpenAI Client** and an **agent reference** (no Application required).

---

## Key points
- Equivalent to the code provided on Foundry portal under `Code` menu, next to the Agent
- Endpoint (project scope): `https://{account}.services.ai.azure.com/api/projects/{project}`
- Auth: **Microsoft Entra ID** (e.g., `DefaultAzureCredential`)
- Invoke via `responses.create(...)` + `extra_body.agent_reference`
- Full **multi‑turn** and **tool execution** within the project boundary
- Ideal for **development, testing, and debugging** before publishing

---

## Minimal setup (Python)
```python
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

PROJECT_URL = "https://<account>.services.ai.azure.com/api/projects/<project-id>"
AGENT_NAME  = "<your-agent-v2-name>"

# Project-scoped client (OAuth)
project_client = AIProjectClient(endpoint=PROJECT_URL, credential=DefaultAzureCredential())

# Resolve the in-project agent
agent = project_client.agents.get(agent_name=AGENT_NAME)

# OpenAI-compatible client bound to the project gateway
openai_client = project_client.get_openai_client()

# Single-turn (stateless example)
resp = openai_client.responses.create(
    input=[{"role": "user", "content": "Tell me what you can help with."}],
    extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
)
print(resp.output_text)

# Multi-turn (client-managed history example)
resp2 = openai_client.responses.create(
    previous_response_id=resp1.id,  # <--- just for Azure OpenAI, otherwise "conversation_id" must be used
    input=[{"role": "user", "content": "What was my previous question?"}],
    extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
)
print(resp2.output_text)

## Prerequisites & Environment Variables

- Python 3.10+  
- `pip install openai azure-identity python-dotenv`
- `pip install --pre azure-ai-projects>=2.0.0b1`
- A **Foundry V2 Project**
- A **Foundry Agent included in the above project** 

Set the following environment variables:
- `FOUNDRY_PROJECT_BASE_URL` → your Microsoft Foundry’s project endpoint base (e.g., `https://<account>.services.ai.azure.com/api/projects/<project>`)
- Optional if you use `DefaultAzureCredential`:  
  `AZURE_TENANT_ID`, `AZURE_CLIENT_ID`, `AZURE_CLIENT_SECRET`  
  or sign in with `az login` to use **Azure CLI credential** [2](https://learn.microsoft.com/en-us/agent-framework/user-guide/agents/agent-types/azure-ai-foundry-models-responses-agent)

# Authentication & Environment setup

In [1]:
import sys
from dotenv import load_dotenv # package python-dotenv
config_path = "../../config"
sys.path.append(config_path)
from auth import acquire_bearer_token, StaticBearerTokenCredential
bearer_token, user = acquire_bearer_token(auth_method="default") # default, cli, device

if not load_dotenv(f"{config_path}/credentials_my.env"):
    print("Environment variables not loaded, cell execution stopped")
else:
    print("Environment variables have been loaded ;-)")

print("Bearer token:", bearer_token[:10], "...")
print(f'User info: {user["name"]}, upn: {user["upn"]}')
# user['raw_claims']

Environment variables have been loaded ;-)
Bearer token: eyJ0eXAiOi ...
User info: Mauro Minella, upn: mauro.minella@MngEnvMCAP883652.onmicrosoft.com


# Libraries and Constants

In [2]:
import os
from IPython.display import Markdown, display
from dotenv import load_dotenv # package python-dotenv
from openai import AzureOpenAI # package openai
from azure.ai.projects import AIProjectClient # pip install (or 'uv add') azure-ai-projects

project_openai_endpoint = os.getenv("AIF_STD_PROJECT_OPENAI_ENDPOINT")
project_endpoint = os.getenv("AIF_STD_PROJECT_ENDPOINT")
deployment_name=os.environ["MODEL_DEPLOYMENT_NAME"]
openai_api_version = os.getenv("AZURE_OPENAI_API_VERSION")
agent_name = os.getenv("AIF_STD_AGENT_NAME")

first_question = "What's the expected temperature in Cavi di Lavagna tomorrow?"
follow_up_question = "What was my previous question?"
project_endpoint, agent_name

('https://aifv2-01-std-foundry.services.ai.azure.com/api/projects/aifv2-01-std-foundryproj01-default',
 'foundryagent-V2-with-bing')

# Create a project client and connect to Microsoft Foundry V2 project

In [3]:
# Retrieve project client
project_client = AIProjectClient(
    endpoint=project_endpoint,
    credential=StaticBearerTokenCredential(bearer_token),
)

In [4]:
# List all the agents, together with their latest version number
for a in list(project_client.agents.list()):
    print(a.id, a["versions"]["latest"]["version"])

foundryagent-V2-with-bing 4
technician01 3
technician02 4


In [5]:
# Retrieve a specific agent
project_client.agents.get(agent_name)

{'object': 'agent', 'id': 'foundryagent-V2-with-bing', 'name': 'foundryagent-V2-with-bing', 'versions': {'latest': {'metadata': {'logo': 'Avatar_Default.svg', 'description': '', 'modified_at': '1772964624', 'microsoft.voice-live.enabled': 'false', 'microsoft.voice-live.configuration': '{"session":{"voice":{"name":"en-US-Ava:DragonHDLatestNeural","type":"azure-standard","temperature":0.8,"rate":"1","isHdVoice":true},"inputAudioTranscription":{"model":"azure-speech"},"turnDetection":{"type":"azure_semantic_vad","endOfUtteranceDetection":null,"removeFillerWords":true},"inputAudioNoiseReduction":null,"inputAudioEchoCancellation":null,"fillerResponse":null,"avatar":null}}'}, 'object': 'agent.version', 'id': 'foundryagent-V2-with-bing:4', 'name': 'foundryagent-V2-with-bing', 'version': '4', 'description': '', 'created_at': 1772964624, 'definition': {'kind': 'prompt', 'model': 'gpt-4.1', 'instructions': 'You are a clever agent that can do WEB searches', 'tools': [{'type': 'bing_grounding', 'b

# Create the OpenAI client (2 ways)

In [6]:
# Retrieve the Azure OpenAI Client, compatible with Response APIs
openai_client = project_client.get_openai_client()
openai_client.base_url

URL('https://aifv2-01-std-foundry.services.ai.azure.com/api/projects/aifv2-01-std-foundryproj01-default/openai/v1/')

In [7]:
# Reference the agent to get a response
resp1 = openai_client.responses.create(
    input=[{"role": "user", "content": first_question}],
    extra_body={"agent_reference": {"name": agent_name, "type": "agent_reference"}},
)

resp1.output_text

'The expected temperature in Cavi di Lavagna tomorrow is a high of 19°C and a low of 15°C, with partly cloudy conditions and possible isolated thunderstorms【4:0†source】.'

In [8]:
# Follow-up question
resp2 = openai_client.responses.create(
    previous_response_id=resp1.id,  # <--- just for Azure OpenAI, otherwise "conversation_id" must be used
    input=follow_up_question,
    extra_body={"agent_reference": {"name": agent_name, "type": "agent_reference"}},
)

print(resp2.output_text)

Your previous question was: "What's the expected temperature in Cavi di Lavagna tomorrow?"
